# Phase 3 Exp 3: 最新アーキテクチャ YOLO26n との比較

## 背景

ここまで Baseline → Exp 1 (解像度UP) → Exp 2 (epochs倍増) と、**ハイパラ最適化**で mAP@0.5 を 0.815 → 0.903 → 0.966 へと改善してきた。Exp 2 の学習曲線は完全に飽和しており、現アーキテクチャ(YOLOv8n)では精度の天井に到達したと判断できる。

Exp 3 では改善軸を変え、**アーキテクチャの新世代化**で更なる精度向上が得られるかを検証する。

## 仮説

**YOLOv8n を最新の YOLO26n に置き換えれば、mAP@0.5 がさらに改善する可能性がある。**

ただし、本タスク(間取り図 6クラス)固有の事情で結果が変わる可能性があり、3つの結末を予想:
- **A(改善)**: YOLO26 の NMS-free 設計がこのタスクに有利、+1〜3pt 改善
- **B(同等)**: 既に mAP 0.966 と上限近く、変化なし、または微細な差
- **C(悪化)**: YOLO26 は CPU推論最適化が主軸であり、GPU高解像度学習では不利という報告あり

**いずれの結果も価値ある知見**:
- A → 最新アーキテクチャの本タスクでの有効性を実証
- B → 「特定ドメインでは新旧モデルの差は縮小する」という観察
- C → 「新しい = 良い、とは限らない」という ML 実務の重要な教訓

## YOLO26 と YOLOv8 のアーキテクチャ差(参考)

| 項目 | YOLOv8 | YOLO26 |
|---|---|---|
| 後処理 | 標準NMS | **NMS-free** (end-to-end) |
| 検出ヘッド | DFL あり | **DFL 削除**(シンプル化) |
| Optimizer (事前学習) | SGD | **MuSGD**(SGD+Muon hybrid) |
| COCO mAP (n) | 37.3 | 40.9 |
| CPU 推論 (ONNX) | 80.4ms | **38.9ms (-51%)** |
| 主な売り | 汎用バランス | **エッジ最適化** |

## 実験設計(Exp 2 との公正な比較のため、モデル以外は完全同一)

| 項目 | Exp 2 | Exp 3 | 変更 |
|---|---|---|---|
| **Model** | yolov8n.pt | **yolo26n.pt** | ⭐ 変更点 |
| imgsz | 1024 | 1024 | 同じ |
| epochs | 100 | 100 | 同じ |
| batch | 8 | 8 | 同じ(OOM ならは縮小) |
| optimizer | auto | auto | 同じ |
| seed | 42 | 42 | 同じ |
| patience | 25 | 25 | 同じ |

## Section 1: 環境セットアップ

In [ ]:
!nvidia-smi

In [ ]:
import os

WORKDIR = "/content/floor-plan-recognition"

if not os.path.exists(WORKDIR):
    !git clone https://github.com/Mao925/floor-plan-recognition.git {WORKDIR}
else:
    %cd {WORKDIR}
    !git pull

%cd {WORKDIR}
!pwd

In [ ]:
!pip install -q ultralytics roboflow python-dotenv

In [ ]:
import torch
import ultralytics

print(f"PyTorch:        {torch.__version__}")
print(f"Ultralytics:    {ultralytics.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Section 2: データ準備

In [ ]:
from google.colab import userdata

try:
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
    print(f"✅ API キー取得成功: {ROBOFLOW_API_KEY[:3]}***{ROBOFLOW_API_KEY[-3:]}")
except Exception as e:
    print(f"❌ エラー: {e}")

In [ ]:
with open('.env', 'w') as f:
    f.write(f'ROBOFLOW_API_KEY={ROBOFLOW_API_KEY}\n')

!python scripts/download_roboflow.py

In [ ]:
!python scripts/prepare_dataset.py

## Section 3: 学習(YOLO26n / imgsz=1024 / epochs=100)

**所要時間予想**: T4 GPU で約 15〜30 分(Exp 2 と同等またはやや遅め)

**注意**: YOLO26 は YOLOv8 より GPU メモリを多く使う可能性あり。OOM が出たら batch を 8→4 に下げる。

In [ ]:
from ultralytics import YOLO

# YOLO26n を使用(自動ダウンロード)
model = YOLO('yolo26n.pt')   # ⭐ 変更点: yolov8n.pt → yolo26n.pt

results = model.train(
    data='data/floorplan_yolo/data.yaml',
    epochs=100,             # Exp 2 と同じ
    imgsz=1024,             # Exp 2 と同じ
    batch=8,                # Exp 2 と同じ(OOM ならは 4 に下げる)
    name='exp3_yolo26n',
    project='runs/detect',
    patience=25,
    save=True,
    plots=True,
    device=0,
    seed=42,
)

print("\n✅ 学習完了")

In [ ]:
# 学習結果のパスを確認
!find runs -name 'best.pt' | head -5

## Section 4: 評価と四者比較(Baseline / Exp 1 / Exp 2 / Exp 3)

In [ ]:
from pathlib import Path

candidates = list(Path('runs').rglob('exp3_yolo26n/weights/best.pt'))
assert candidates, "best.pt が見つかりません"
best_pt = candidates[0]
results_dir = best_pt.parent.parent
print(f"学習結果フォルダ: {results_dir}")
print(f"ベストモデル:     {best_pt}")

In [ ]:
# 学習曲線などを表示
from IPython.display import Image, display

for img_name in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
    img_path = results_dir / img_name
    if img_path.exists():
        print(f"\n=== {img_name} ===")
        display(Image(str(img_path)))

In [ ]:
# test セットで最終評価
best_model = YOLO(str(best_pt))

test_metrics = best_model.val(
    data='data/floorplan_yolo/data.yaml',
    split='test',
    imgsz=1024,
    name='exp3_test_eval',
    project='runs/detect',
)

print("\n=== Test セット全体メトリクス (Exp 3: YOLO26n) ===")
print(f"mAP@0.5:        {test_metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95:   {test_metrics.box.map:.4f}")
print(f"Precision:      {test_metrics.box.mp:.4f}")
print(f"Recall:         {test_metrics.box.mr:.4f}")

In [ ]:
# 四者比較表(Baseline / Exp 1 / Exp 2 / Exp 3)
import pandas as pd

class_names = ['door', 'shower', 'sink', 'staircase', 'toilet', 'window']

# これまでの実験結果(README に記録した値をハードコード)
baseline_ap50 = {
    'door': 0.9862, 'shower': 0.4174, 'sink': 0.8231,
    'staircase': 0.7045, 'toilet': 0.9656, 'window': 0.9924
}
exp1_ap50 = {
    'door': 0.9873, 'shower': 0.7297, 'sink': 0.9617,
    'staircase': 0.7591, 'toilet': 0.9855, 'window': 0.9928
}
exp2_ap50 = {
    'door': 0.9897, 'shower': 0.9246, 'sink': 0.9887,
    'staircase': 0.9070, 'toilet': 0.9950, 'window': 0.9912
}

exp3_ap50 = {}
for i, name in enumerate(class_names):
    ap = float(test_metrics.box.ap50[i]) if i < len(test_metrics.box.ap50) else 0
    exp3_ap50[name] = ap

df = pd.DataFrame({
    'Baseline':       [baseline_ap50[c] for c in class_names],
    'Exp 1 (1024)':   [exp1_ap50[c]     for c in class_names],
    'Exp 2 (100ep)':  [exp2_ap50[c]     for c in class_names],
    'Exp 3 (YOLO26)': [exp3_ap50[c]     for c in class_names],
}, index=class_names)
df['Δ (Exp3 - Exp2)'] = df['Exp 3 (YOLO26)'] - df['Exp 2 (100ep)']
df['Verdict'] = df['Δ (Exp3 - Exp2)'].apply(
    lambda x: '↑ 改善' if x > 0.01 else ('↓ 悪化' if x < -0.01 else '− 変化なし')
)

print("=" * 100)
print("クラス別 mAP@0.5: 四者比較")
print("=" * 100)
print(df.to_string(float_format=lambda x: f'{x:.4f}' if isinstance(x, float) else str(x)))

# 全体比較
baseline_overall = 0.815
exp1_overall = 0.9027
exp2_overall = 0.9660
print(f"\n=== Overall mAP@0.5 四者比較 ===")
print(f"  Baseline (YOLOv8n, 640,  50ep):   {baseline_overall:.4f}")
print(f"  Exp 1    (YOLOv8n, 1024, 50ep):   {exp1_overall:.4f}   (vs Baseline: {exp1_overall - baseline_overall:+.4f})")
print(f"  Exp 2    (YOLOv8n, 1024, 100ep):  {exp2_overall:.4f}   (vs Exp 1:    {exp2_overall - exp1_overall:+.4f})")
print(f"  Exp 3    (YOLO26n, 1024, 100ep):  {test_metrics.box.map50:.4f}   (vs Exp 2:    {test_metrics.box.map50 - exp2_overall:+.4f})")

In [ ]:
# 推論サンプル
import random

test_images = sorted(Path('data/floorplan_yolo/test/images').glob('*.jpg'))
random.seed(42)
samples = random.sample(test_images, min(4, len(test_images)))

predict_results = best_model.predict(
    source=[str(p) for p in samples],
    imgsz=1024,
    save=True,
    project='runs/detect',
    name='exp3_samples',
    conf=0.25,
)

pred_dir = list(Path('runs').rglob('exp3_samples'))[0]
for img_path in sorted(pred_dir.glob('*.jpg')):
    print(f"\n=== {img_path.name} ===")
    display(Image(str(img_path)))

## Section 5: 結果の保存

In [ ]:
import shutil

src = str(results_dir)
out_zip = '/content/exp3_yolo26n_results.zip'
shutil.make_archive(out_zip.replace('.zip', ''), 'zip', src)
print(f"✅ Zip 作成完了: {out_zip}")
!ls -lh {out_zip}

In [ ]:
from google.colab import files
files.download(out_zip)

---

## 検証結果のまとめ(実験完了後にここに記入する)

### 仮説
Exp 2 と同条件で、モデルを YOLOv8n → YOLO26n に変更すると、mAP@0.5 が改善する可能性がある。

### 結果(↑ 上のセル出力を見て手動で記入)
- 全体 mAP@0.5: 0.966 (Exp 2) → ?
- 学習時間: 15分 (Exp 2) → ?

### 結果の分類
- A(改善): Δ > +0.01 → 最新アーキテクチャの有効性を実証
- B(同等): -0.01 < Δ < +0.01 → 「特定タスクでは差が縮小する」観察
- C(悪化): Δ < -0.01 → 「新しい = 良い、とは限らない」教訓

### 解釈すべきポイント
- 学習時間の変化(GPU負荷)
- クラス別の傾向(NMS-free 化が小物体に効くか?)
- パラメータ数の比較(YOLO26n vs YOLOv8n)